# Examine Relationship Between Combine Results And DB Separation During Go/Post/Corner Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We've found previously that our engineered Combine metrics derived from Combine player-tracking data did not have any statistically significant associations with a DBs ability to minimize separation at burst during go/post/corner routes. 

Here we will make sure that there also aren't associations between the DBs ability to minimize separation and general Combine results (such as 40-yard dash time, 3 Cone drill time, etc.)

Predictors: General Combine Results

Outcome: DBs Minimizing Separation At Burst During Go/Post/Corner Routes

**<span style="color: gold;">FINDINGS</span>**

No meaningful relationships between general Combine results and a DBs ability to minimize separation at burst on go/post/corner routes.

**<span style="color: gold;">NOTES</span>**

Since we need to craete outcome variables related to separation and since we need to know who is guarding who, we must use player-tracking data from the 2026 Big Data Bowl. That data only includes 2023 Regular season games. Therefore, our analysis will only include 2023 rookies during 2023 Regular season time. 

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================
param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.append(str(param_path))
import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["GO", "POST", "CORNER"]

<br><br/>
#### DB Separation Data From *11_BDB26_DB_Player_Tracking.ipynb*

In [ ]:
# =========================================
# DB Separation Data Among 2023 Rookies
# =========================================

db_separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "11_2023_Rookie_DB_Player_Time_Level_Separation_Data.csv"))

# -----------------------------------------
# Identify Players Of Interest
# -----------------------------------------

players_of_interest = (
    db_separation_df
    .select("defender_nfl_id")
    .unique()
    .to_series()
    .to_list()
)

# ----------------------------------------
# Only Look At Specified Routes
# ----------------------------------------

db_separation_df = (
    db_separation_df
    .filter(pl.col("route").is_in(ROUTES_OF_INTEREST))
)

# ----------------------------------------
# Only Look At Man Coverage Situations
# ----------------------------------------

# db_separation_df = (
#     db_separation_df
#     .filter(pl.col("team_coverage_man_zone") == "MAN_COVERAGE")
# )

print("Separation Data:")
print(f"    - Dimensions: {db_separation_df.shape}")
print(f"    - Number of Rookie DBs: {db_separation_df.select("defender_nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {db_separation_df.select("player_play_id").n_unique()}")
db_separation_df

<br><br/>
#### Combine Features Data

In [ ]:
# =======================================
# Combine Results Data
# =======================================

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
)
print("Combine Results Data:")
print(f"    - Dimensions: {combine_results_df.shape}")
print(f"    - Number of Players: {combine_results_df.select("nfl_id").n_unique()}")
combine_results_df

<br><br/>
## Data Cleaning
***

#### Get Data On Cuts And Separation At Cut From Targeted Hitch/In Routes

In [ ]:
# ==============================================================================
# Extract In-Game Acceleration and Separation for Go Routes
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Calculate Downfield Kinematics and Filter for Vertical Push
# ------------------------------------------------------------------------------
go_kinematics_df = (
    db_separation_df
    .sort(["player_play_id", "frame_id"])
    .with_columns(
        # Calculate raw directional velocities (dt = 0.1s at 10Hz)
        vx = (pl.col("wr_x") - pl.col("wr_x").shift(1).over("player_play_id")) / 0.1,
        vy = (pl.col("wr_y") - pl.col("wr_y").shift(1).over("player_play_id")) / 0.1,
    )
    .with_columns(
        # Speed magnitude
        s = (pl.col("vx")**2 + pl.col("vy")**2).sqrt()
    )
    .with_columns(
        # True linear acceleration
        linear_accel = (pl.col("s") - pl.col("s").shift(1).over("player_play_id")) / 0.1
    )
    .with_columns(
        # Isolate true downfield velocity based on play direction
        v_downfield = pl.when(pl.col("play_direction") == "right")
                      .then(pl.col("vx"))
                      .otherwise(-pl.col("vx")),
        # Calculate relative distance from the Line of Scrimmage (LOS)
        # Assumes absolute distance downfield from where the play started
        yards_midfield = pl.when(pl.col("play_direction") == "right")
                         .then(pl.col("wr_x") - pl.col("absolute_yardline_number"))
                         .otherwise(pl.col("absolute_yardline_number") - pl.col("wr_x"))
    )
)

# ------------------------------------------------------------------------------
# 2. Isolate Key Frames: The Snap (Release) and Peak Downfield Burst
# ------------------------------------------------------------------------------
# For Go routes, we want to find the frame where they hit their absolute highest 
# acceleration gear *after* they've advanced at least 5 yards past the LOS. 
# This prevents capturing short hand-fights at the line and focuses on deep vertical burn.
go_burst_frames_df = (
    go_kinematics_df
    .filter(
        (pl.col("linear_accel").is_not_null()) & 
        (pl.col("yards_midfield") > 5.0) # Clear the initial line-of-scrimmage jam
    )
    .group_by("player_play_id")
    .agg([
        # Find the frame where the WR hits maximum acceleration downfield
        pl.col("frame_id").filter(pl.col("linear_accel") == pl.col("linear_accel").max()).first().alias("peak_burst_frame_id"),
        # Also capture the snap/start frame of the play for baseline release tracking
        pl.col("frame_id").min().alias("snap_frame_id")
    ])
)

# ------------------------------------------------------------------------------
# 3. Extract Separation at Release and Peak Vertical Burst
# ------------------------------------------------------------------------------
final_go_separation_df = (
    go_kinematics_df
    .join(go_burst_frames_df, on="player_play_id", how="inner")
    .filter(
        (pl.col("frame_id") == pl.col("snap_frame_id")) |                          # At the line
        (pl.col("frame_id") == (pl.col("snap_frame_id") + 10)) |                    # 1.0s after snap (Release Separation)
        (pl.col("frame_id") == pl.col("peak_burst_frame_id")) |                    # Moment of peak deep acceleration
        (pl.col("frame_id") == (pl.col("peak_burst_frame_id") + 5))                 # 0.5s after peak acceleration
    )
    .group_by("player_play_id")
    .agg([
        pl.col("defender_nfl_id").first(),
        pl.col("defender_name").first(),
        pl.col("defender_position").first(),
        
        # 1. Initial Release Separation: How well they stack the CB in the first 10 frames (1 second)
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("snap_frame_id") + 10)).first().alias("separation_at_release"),
        
        # 2. Deep Burst Separation: Distance to CB the exact moment they turn on their highest gear
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == pl.col("peak_burst_frame_id")).first().alias("separation_at_peak_burst"),
        
        # 3. Post-Burst Separation: Did the acceleration actually burn the CB 0.5 seconds later?
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("peak_burst_frame_id") + 5)).first().alias("separation_post_burst")
    ])
    .with_columns(
        # The ultimate metric: Net separation generated *by* the downfield burst
        go_separation_generated = pl.col("separation_post_burst") - pl.col("separation_at_peak_burst")
    )
)

# ------------------------------------------------------------------------------
# 4. Roll up to Player Averages
# ------------------------------------------------------------------------------
db_closeness_at_burst_df = (
    final_go_separation_df
    .group_by("defender_nfl_id")
    .agg([
        pl.col("defender_name").first(),
        pl.col("defender_position").first(),
        pl.col("separation_at_release").mean().alias("avg_release_separation"),
        pl.col("separation_at_peak_burst").mean().alias("avg_sep_at_burst_point"),
        pl.col("go_separation_generated").mean().alias("avg_go_separation_generated")
    ])
)

db_closeness_at_burst_df


<br><br/>
#### Add Combine Metrics To Player Separation Data

In [ ]:
analysis_df = (
    db_closeness_at_burst_df
    .join(
        combine_results_df, 
        left_on="defender_nfl_id", 
        right_on="nfl_id", 
        how = "inner"
    )
)

print("Final Analysis Dataset:")
print(f"    - Number of WRs: {analysis_df.select("defender_nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine Associations
***

We've previously seen that there are no associations between our engineered Combine features and a DBs ability to minimize separation at burst during go/post/corner routes (*avg_go_separation_generated*). We will now check if there are any associations when we use general Combine results as the predictors.

<br><br/>
#### General Combine Results vs Separation Lost

In [ ]:
predictor = "ngs_athleticism_score" 
outcome = "avg_go_separation_generated"

fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols"
)

fig.show()

In [ ]:
# ==========================================
# Univariate Model
# ==========================================

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())